In [ ]:
# Load environment variables and set up auto-reload
from dotenv import load_dotenv
load_dotenv()
from utils import patch_mcp_stdio_for_jupyter
patch_mcp_stdio_for_jupyter()

In [5]:
# Simple MCP Example
import os

# Must run before langchain_mcp_adapters import (Jupyter + Windows stdio fix)
from utils import patch_mcp_stdio_for_jupyter
patch_mcp_stdio_for_jupyter()

from langchain_mcp_adapters.client import MultiServerMCPClient
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

# Get the absolute path to our sample research docs
sample_docs_path = os.path.abspath("./files/")
console.print(f"[bold blue]Sample docs path:[/bold blue] {sample_docs_path}")

# Check if the directory exists
if os.path.exists(sample_docs_path):
    console.print(f"[green]✓ Directory exists with files:[/green] {os.listdir(sample_docs_path)}")
else:
    console.print("[red]✗ Directory does not exist![/red]")

# MCP Client configuration - filesystem server for local document access
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": [
            "-y",  # Auto-install if needed
            "@modelcontextprotocol/server-filesystem",
            sample_docs_path
        ],
        "transport": "stdio"
    }
}

console.print(Panel("[bold yellow]Creating MCP client...[/bold yellow]", expand=False))
client = MultiServerMCPClient(mcp_config)
console.print("[green]✓ MCP client created successfully![/green]")

# Test getting tools
console.print(Panel("[bold yellow]Getting tools...[/bold yellow]", expand=False))
tools = await client.get_tools()

# Create a rich table for tool display
table = Table(title="Available MCP Tools", show_header=True, header_style="bold magenta")
table.add_column("Tool Name", style="cyan", width=25)
table.add_column("Description", style="white", width=80)

for tool in tools:
    # Truncate long descriptions for better display
    description = tool.description[:77] + "..." if len(tool.description) > 80 else tool.description
    table.add_row(tool.name, description)

console.print(table)
console.print(f"[bold green]✓ Successfully retrieved {len(tools)} tools from MCP server[/bold green]")

Sample docs path: e:\Learn\Langchain_academy\deepagent\notebooks\files

✓ Directory exists with files: ['coffee_shops_sf.md']

╭────────────────────────╮
│ Creating MCP client... │
╰────────────────────────╯

✓ MCP client created successfully!

╭──────────────────╮
│ Getting tools... │
╰──────────────────╯

                                              Available MCP Tools                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tool Name                 ┃ Description                                                                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ read_file                 │ Read the complete contents of a file as text. DEPRECATED: Use read_text_file ... │
│ read_text_file            │ Read the complete contents of a file from the file system as text. Handles va... │
│ read_media_file           │ Read an image or audio file. Returns the base64 encoded data and MIME type. O... │
│ read_multiple_files       │ Read the contents of multiple files simultaneously. This is more efficient th... │
│ write_file                │ Create a new file or completely overwrite an existing file with new content. ... │
│ edit_file                 │ Make line-based edits to a text file. Each edit replaces exact line sequences... │
│ create_directory          │ Create a new directory or ensure a directory exists. Can create multiple nest... │
│ list_directory            │ Get a detailed listing of all files and directories in a specified path. Resu... │
│ list_directory_with_sizes │ Get a detailed listing of all files and directories in a specified path, incl... │
│ directory_tree            │ Get a recursive tree view of files and directories as a JSON structure. Each ... │
│ move_file                 │ Move or rename files and directories. Can move files between directories and ... │
│ search_files              │ Recursively search for files and directories matching a pattern. The patterns... │
│ get_file_info             │ Retrieve detailed metadata about a file or directory. Returns comprehensive i... │
│ list_allowed_directories  │ Returns the list of directories that this server is allowed to access. Subdir... │
└───────────────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

✓ Successfully retrieved 14 tools from MCP server